# Extending the Warehouse Order Processing System (OMS v2)

Orders now arrive in **batches** (e.g. all orders received in one day). The program:

1. Collects `n` orders, storing each one as its own list and appending it to a master list `all_orders`.
2. Loops through `all_orders`, applies the business rules to each order and prints an individual summary.
3. Prints an **end-of-day report** built only from `all_orders` and the lists filled while processing.

## Business rules (same as before)

| Rule | Condition |
|---|---|
| Order ACCEPTED | order value > ₹500 **and** at least 1 item **and** distance ≤ 100 km |
| FREE shipping | order value ≥ ₹2000 **or** premium member |
| PRIORITY processing | marked as priority **and** order value ≥ ₹1500 |
| SPECIAL handling | fragile item **or** 10 or more items |

```
Shipping cost = ₹50 base charge + ₹8 for every kilometre
```

## Step 1 – Collect multiple orders into a list

Each order is stored as:

```
[customer_id, order_value, num_items, distance_km, is_premium, is_priority, is_fragile]
```

For yes/no questions, type `Yes` or `No`.

In [ ]:
# Master list that will hold every order in the batch
all_orders = []

n = int(input("How many orders do you want to enter? "))

for i in range(n):
    print("--- Order", i + 1, "of", n, "---")
    customer_id = input("Enter Customer ID: ")
    order_value = float(input("Enter Order value (₹): "))
    num_items = int(input("Enter Number of items: "))
    distance_km = float(input("Enter Delivery distance (km): "))
    premium_input = input("Is the customer a premium member? (Yes/No): ")
    priority_input = input("Does the order require priority processing? (Yes/No): ")
    fragile_input = input("Does the order contain a fragile item? (Yes/No): ")

    # Convert Yes/No answers to True/False
    is_premium = premium_input.strip().lower() in ["yes", "y"]
    is_priority = priority_input.strip().lower() in ["yes", "y"]
    is_fragile = fragile_input.strip().lower() in ["yes", "y"]

    # Store this order as its own list and add it to the master list
    order = [customer_id, order_value, num_items, distance_km, is_premium, is_priority, is_fragile]
    all_orders.append(order)

print()
print(len(all_orders), "order(s) stored in all_orders.")

### (Optional) Sample batch for testing

**Skip this cell if you entered your own orders above** — running it replaces `all_orders` with a 5-order test batch so Steps 2 and 3 can be checked without typing.

In [ ]:
# Sample batch: [customer_id, order_value, num_items, distance_km, is_premium, is_priority, is_fragile]
all_orders = [
    ["C104", 2450, 6, 18, True, False, True],     # example order from the original assignment
    ["C201", 1800, 12, 40, False, True, False],   # accepted, paid shipping, priority, 10+ items
    ["C305", 450, 2, 10, False, False, False],    # rejected: value not above ₹500
    ["C412", 3200, 3, 120, True, True, True],     # rejected: distance over 100 km
    ["C518", 900, 1, 25.5, False, False, False],  # accepted, paid shipping
]

print(len(all_orders), "sample order(s) loaded into all_orders.")

## Step 2 – Process every order in the batch

For each order the results are printed and also saved in parallel lists (`statuses`, `shipping_statuses`, `shipping_costs`, `processings`, `handlings`) so the end-of-day report can use them. Position `i` in each list belongs to `all_orders[i]`.

In [ ]:
# Lists built while processing (index i matches all_orders[i])
statuses = []
shipping_statuses = []
shipping_costs = []
processings = []
handlings = []

for order in all_orders:
    # Unpack the order list into named variables
    customer_id, order_value, num_items, distance_km, is_premium, is_priority, is_fragile = order

    # Calculate shipping cost: ₹50 base + ₹8 per km
    shipping_cost = 50 + (8 * distance_km)

    # Rule 1: order can be processed only when value > 500, at least 1 item, and distance <= 100 km
    if order_value > 500 and num_items >= 1 and distance_km <= 100:
        status = "ACCEPTED"
    else:
        status = "REJECTED"

    # Rule 2: free shipping when order value >= 2000 or customer is a premium member
    if order_value >= 2000 or is_premium:
        shipping_status = "FREE"
    else:
        if shipping_cost == int(shipping_cost):
            shipping_status = "₹" + str(int(shipping_cost))
        else:
            shipping_status = "₹" + str(shipping_cost)

    # Rule 3: priority processing only when marked priority AND value >= 1500
    if is_priority and order_value >= 1500:
        processing = "PRIORITY"
    else:
        processing = "STANDARD"

    # Rule 4: special handling when fragile item or 10 or more items
    if is_fragile or num_items >= 10:
        handling = "SPECIAL"
    else:
        handling = "NORMAL"

    # Save the results for the end-of-day report
    statuses.append(status)
    shipping_statuses.append(shipping_status)
    shipping_costs.append(shipping_cost)
    processings.append(processing)
    handlings.append(handling)

    # Display the order-processing summary
    print("===================================")
    print("ORDER PROCESSING RESULT")
    print("===================================")
    print("Customer \t:", customer_id)
    if order_value == int(order_value):
        print("Order value \t: ₹" + str(int(order_value)))
    else:
        print("Order value \t: ₹" + str(order_value))
    print("Items \t:", num_items)
    if distance_km == int(distance_km):
        print("Distance \t:", str(int(distance_km)) + " km")
    else:
        print("Distance \t:", str(distance_km) + " km")
    print("Status \t:", status)
    print("Shipping \t:", shipping_status)
    print("Processing \t:", processing)
    print("Handling \t:", handling)
    print("===================================")
    print()

## Step 3 – End-of-day report

Uses only `all_orders` and the lists built in Step 2.

- **Revenue** and **shipping collected** count only ACCEPTED orders (shipping only when it was not FREE).
- **Highest-value order**, **special handling** and **priority processing** are counted across the whole batch, matching the per-order summaries above.

In [ ]:
total_orders = len(all_orders)
accepted_count = 0
rejected_count = 0
total_revenue = 0
total_shipping = 0
special_count = 0
priority_count = 0

# Start with the first order as the highest, then compare with the rest
highest_index = 0

for i in range(total_orders):
    order_value = all_orders[i][1]

    if statuses[i] == "ACCEPTED":
        accepted_count += 1
        total_revenue += order_value
        # Shipping is collected only if the accepted order did not get free shipping
        if shipping_statuses[i] != "FREE":
            total_shipping += shipping_costs[i]
    else:
        rejected_count += 1

    if order_value > all_orders[highest_index][1]:
        highest_index = i

    if handlings[i] == "SPECIAL":
        special_count += 1

    if processings[i] == "PRIORITY":
        priority_count += 1

# Show whole numbers without ".0", like the per-order summaries
if total_revenue == int(total_revenue):
    total_revenue = int(total_revenue)
if total_shipping == int(total_shipping):
    total_shipping = int(total_shipping)

print("===========================================")
print("END-OF-DAY REPORT")
print("===========================================")
print("Total orders received \t:", total_orders)
print("Orders ACCEPTED \t:", accepted_count)
print("Orders REJECTED \t:", rejected_count)
print("Total revenue (accepted) \t: ₹" + str(total_revenue))
print("Total shipping collected \t: ₹" + str(total_shipping))

if total_orders > 0:
    highest_order = all_orders[highest_index]
    highest_value = highest_order[1]
    if highest_value == int(highest_value):
        highest_value = int(highest_value)
    print("Highest-value order \t:", highest_order[0], "(₹" + str(highest_value) + ")")
else:
    print("Highest-value order \t: none (no orders entered)")

print("Special handling orders \t:", special_count)
print("Priority processing orders \t:", priority_count)
print("===========================================")